# 🌍 GeoVault — Interactive Data Stewardship Validator

Upload your research dataset, validate it against FAIR-aligned stewardship
policies, and download compliance reports.

### What this notebook does

1. 🖥️ Prints an **environment banner** — toolkit, version, and exactly what to upload
2. 📋 Shows a **live checklist of required files** with the exact expected
   CSV headers, so you always know what to prepare
3. ⬇️ Provides **downloadable templates** — one per registered dataset —
   pre-filled with those headers
4. 📤 Lets you **upload** your dataset files (CSV, YAML, LICENSE) — every
   upload is **previewed automatically**
5. ✅ **Validates** metadata, licensing, and cryptographic checksums — the
   audit **runs automatically** when the cell executes
6. 💾 **Auto-saves** `compliance_report.json`, `compliance_report.txt`,
   `integrity.sha256` and `audit_summary.html` after every audit (on Colab
   the JSON report downloads automatically) so `dashboard.html` can be
   refreshed in one commit

> ▶️ **Auto-run ready:** *Runtime → Run all* completes end-to-end. With no
> uploads the notebook validates the bundled demo dataset (`sentinel2_l2a`),
> so you can watch every step pass green before plugging in your own data.

> Tip: every check mirrors the repository's `geovault_core.inspector` engine, so
> a dataset that passes here will pass the CI audit too. Run the cells in
> order, top to bottom.

---


## ⚙️ Step 1 — Setup & Dependencies

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyyaml ipywidgets

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

import csv
import hashlib
import io
import json
import re
from datetime import datetime
from pathlib import Path

import yaml
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

print("✅ Dependencies loaded")
print("📅 Session started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("🏠 Environment:", "Google Colab" if IN_COLAB else "local Jupyter (downloads are saved next to the notebook)")

# 🖥️ Environment banner: what this toolkit is and what it expects from you.
print()
print("=" * 62)
print("  🌍 GeoVault v1.3.0 — Remote Sensing, GIS & ML Data Stewardship Pipeline")
print("  Domain   : Remote Sensing / GIS / Machine Learning")
print("  Formats  : GeoTIFF, NetCDF, CSV · Standards: FAIR, ISO 19115, DataCite")
print("  Uploads  : record.yaml + data.csv + LICENSE.txt (Step 3)")
print("  Header   : scene_id,acq_date,sensor,cloud_pct,crs,resolution_m")
print("=" * 62)


### ☁️ Step 1b — Optional: mount Google Drive

Large scene files (GeoTIFF, NetCDF) often live on Drive. Set
`MOUNT_DRIVE = True` **before running this cell** to attach your Drive;
leave it `False` (default) to skip — every other step works without it.

In [ ]:
MOUNT_DRIVE = False  # ← set True to attach your Google Drive for large rasters

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted at /content/drive")
else:
    reason = "" if IN_COLAB else " or not on Colab"
    print("⏭️ Drive mount skipped (MOUNT_DRIVE=False" + reason + ").")


## 📋 Step 2 — Download Templates

Before uploading your data, download these templates. They contain the
**exact headers and structure** the validator expects — `data_template.csv`
already carries the GeoVault header row (`scene_id,acq_date,sensor,cloud_pct,crs,resolution_m`), so you only
replace the sample rows with your measurements. Every additional registered
dataset has its own template below it.

In [ ]:
TEMPLATES = {
    "data.csv (dataset template)": {
        "filename": "data_template.csv",
        "description": ("CSV with the exact headers the validator expects "
                        "(columns: scene_id,acq_date,sensor,cloud_pct,crs,resolution_m). Keep the header row and "
                        "replace the sample rows with your data."),
        "content": (
            "scene_id,acq_date,sensor,cloud_pct,crs,resolution_m\n"
            "YOUR_SCENE_ID,YOUR_ACQ_DATE,YOUR_SENSOR,YOUR_CLOUD_PCT,YOUR_CRS,YOUR_RESOLUTION_M\n"
            "YOUR_SCENE_ID,YOUR_ACQ_DATE,YOUR_SENSOR,YOUR_CLOUD_PCT,YOUR_CRS,YOUR_RESOLUTION_M\n"
        ),
    },
    "landsat9_c2l2 \u2014 CSV template": {
        "filename": "template_landsat9_c2l2.csv",
        "description": "Upload template for Landsat 9 Collection 2 Scene Register \u2014 keep the header row (columns: scene_id,acq_date,wrs_path,wrs_row,cloud_pct,crs), replace the sample rows.",
        "content": "scene_id,acq_date,wrs_path,wrs_row,cloud_pct,crs\nYOUR_SCENE_ID,YOUR_ACQ_DATE,YOUR_WRS_PATH,YOUR_WRS_ROW,YOUR_CLOUD_PCT,YOUR_CRS\nYOUR_SCENE_ID,YOUR_ACQ_DATE,YOUR_WRS_PATH,YOUR_WRS_ROW,YOUR_CLOUD_PCT,YOUR_CRS\n",
    },
    "uav_orthomosaic \u2014 CSV template": {
        "filename": "template_uav_orthomosaic.csv",
        "description": "Upload template for UAV Orthomosaic Survey Register \u2014 keep the header row (columns: flight_id,acq_date,sensor,gsd_cm,crs,area_ha), replace the sample rows.",
        "content": "flight_id,acq_date,sensor,gsd_cm,crs,area_ha\nYOUR_FLIGHT_ID,YOUR_ACQ_DATE,YOUR_SENSOR,YOUR_GSD_CM,YOUR_CRS,YOUR_AREA_HA\nYOUR_FLIGHT_ID,YOUR_ACQ_DATE,YOUR_SENSOR,YOUR_GSD_CM,YOUR_CRS,YOUR_AREA_HA\n",
    },
    "dem_cop30 \u2014 CSV template": {
        "filename": "template_dem_cop30.csv",
        "description": "Upload template for Copernicus DEM 30 m Tile Register \u2014 keep the header row (columns: tile_id,source,resolution_m,crs,version), replace the sample rows.",
        "content": "tile_id,source,resolution_m,crs,version\nYOUR_TILE_ID,YOUR_SOURCE,YOUR_RESOLUTION_M,YOUR_CRS,YOUR_VERSION\nYOUR_TILE_ID,YOUR_SOURCE,YOUR_RESOLUTION_M,YOUR_CRS,YOUR_VERSION\n",
    },
    "landcover_labels \u2014 CSV template": {
        "filename": "template_landcover_labels.csv",
        "description": "Upload template for Land-Cover Training Labels v2 \u2014 keep the header row (columns: plot_id,lat,lon,obs_date,landcover,sample_source), replace the sample rows.",
        "content": "plot_id,lat,lon,obs_date,landcover,sample_source\nYOUR_PLOT_ID,YOUR_LAT,YOUR_LON,YOUR_OBS_DATE,YOUR_LANDCOVER,YOUR_SAMPLE_SOURCE\nYOUR_PLOT_ID,YOUR_LAT,YOUR_LON,YOUR_OBS_DATE,YOUR_LANDCOVER,YOUR_SAMPLE_SOURCE\n",
    },
    "gps_survey \u2014 CSV template": {
        "filename": "template_gps_survey.csv",
        "description": "Upload template for Ground-Truth GPS Survey Points \u2014 keep the header row (columns: point_id,lat,lon,elev_m,visit_date,operator), replace the sample rows.",
        "content": "point_id,lat,lon,elev_m,visit_date,operator\nYOUR_POINT_ID,YOUR_LAT,YOUR_LON,YOUR_ELEV_M,YOUR_VISIT_DATE,YOUR_OPERATOR\nYOUR_POINT_ID,YOUR_LAT,YOUR_LON,YOUR_ELEV_M,YOUR_VISIT_DATE,YOUR_OPERATOR\n",
    },
    "record.yaml (metadata template)": {
        "filename": "record_template.yaml",
        "description": "FAIR metadata record. Every field is REQUIRED and version must be MAJOR.MINOR.PATCH.",
        "content": (
            "asset_id: my_dataset_001\n"
            'title: "My Research Dataset 2026"\n'
            'version: "1.0.0"\n'
            'custodian: "Your Name"\n'
            "license_ref: CC-BY-4.0\n"
            "abstract: >\n"
            "  Brief description of what this dataset contains,\n"
            "  how it was collected, and its purpose.\n"
            "keywords: [research, dataset, example]\n"
            "format: CSV\n"
            "provenance:\n"
            '  source: "Laboratory instrument X"\n'
            '  processing: "Raw data, no transformations applied"\n'
        ),
    },
    "LICENSE.txt (license template)": {
        "filename": "LICENSE_template.txt",
        "description": "Plain-text license. Choose from: MIT, CC-BY-4.0, CC0-1.0, CC-BY-SA-4.0.",
        "content": (
            "This dataset is licensed under the Creative Commons Attribution 4.0\n"
            "International License (CC-BY-4.0).\n"
            "\n"
            "You are free to:\n"
            "  - Share — copy and redistribute the material in any medium or format\n"
            "  - Adapt — remix, transform, and build upon the material for any purpose\n"
            "\n"
            "Under the following terms:\n"
            "  - Attribution — you must give appropriate credit to the original author.\n"
            "\n"
            "Full license text: https://creativecommons.org/licenses/by/4.0/\n"
        ),
    },
}


def save_bytes(filename, text):
    Path(filename).write_text(text, encoding="utf-8")
    if IN_COLAB and files is not None:
        files.download(filename)


print("📥 Click a button to download a template:\n")

for label, template in TEMPLATES.items():
    box = widgets.HTML(
        "<div style='margin:10px 0;padding:10px;background:#f6f8fa;"
        "border-left:4px solid #0b1d3a;border-radius:4px;'>"
        "<strong>" + label + "</strong><br><em>" + template["description"] + "</em></div>"
    )
    button = widgets.Button(description="⬇️ Download " + template["filename"], button_style="primary")

    def on_download(_clicked, template=template):
        save_bytes(template["filename"], template["content"])
        print("✅ Downloaded:", template["filename"])

    button.on_click(on_download)
    display(box, button)

print("\n💡 Fill out the templates with your data, then upload them in Step 3.")


## 📤 Step 3 — Upload Your Dataset Files

Upload **all required files** for one dataset — the live checklist below the
upload widget tracks exactly what is still missing:

| # | Required file | Purpose | Expected header / content |
| - | ------------- | ------- | ------------------------- |
| 1 | `record.yaml` | metadata record | `asset_id`, `title`, `version`, `custodian`, `license_ref`, `abstract` |
| 2 | `data.csv` | dataset contents | `scene_id,acq_date,sensor,cloud_pct,crs,resolution_m` |
| 3 | `LICENSE.txt` | license terms | plain text (MIT, CC-BY-4.0, CC0-1.0, CC-BY-SA-4.0) |
| ⏭️ | `integrity.sha256` | auto-generated | built for you in Step 5 |

Files may keep any name as long as the type is recognizable (`…_data.csv`,
`my_record.yaml`, …). CSV headers should match the template — intentional
extra columns only raise a warning, never a failure.

In [ ]:
UPLOADED_FILES = {}

upload_widget = widgets.FileUpload(accept="", multiple=True, description="📁 Select Files")
preview_button = widgets.Button(description="🔍 Preview Uploaded Files", button_style="success")
preview_output = widgets.Output()
checklist = widgets.HTML()

REQUIRED_FILES = {
    # Required file -> (purpose, expected header / content). The live
    # checklist below tracks every file so it is always obvious what still
    # needs to be uploaded.
    "record.yaml": ("metadata record",
                    "asset_id, title, version, custodian, license_ref, abstract"),
    "data.csv": ("dataset contents", "scene_id,acq_date,sensor,cloud_pct,crs,resolution_m"),
    "LICENSE.txt": ("license terms",
                    "plain text — MIT, CC-BY-4.0, CC0-1.0 or CC-BY-SA-4.0"),
}
# CSV header the validator expects (a warning — never a failure — when yours
# differs on purpose).
EXPECTED_HEADERS = {"data.csv": "scene_id,acq_date,sensor,cloud_pct,crs,resolution_m"}
# Housekeeping files that must never enter a checksum manifest.
SKIP_CHECKSUM_FILES = {"record.yaml", "license.txt", "readme.md", "changelog.md",
                       ".ds_store", "thumbs.db", "integrity.sha256"}


def _decode(raw):
    if isinstance(raw, (bytes, bytearray, memoryview)):
        data = bytes(raw)
        try:
            return data.decode("utf-8")
        except UnicodeDecodeError:
            return data.decode("latin-1")
    return str(raw)


def _find_uploaded(needle):
    for name in UPLOADED_FILES:
        if needle.lower() in name.lower():
            return name
    return None


def _header_note(required, name):
    expected = EXPECTED_HEADERS.get(required)
    if not expected or not required.lower().endswith(".csv"):
        return ""
    header = (UPLOADED_FILES[name].splitlines() or [""])[0].strip()
    if [cell.strip().lower() for cell in header.split(",")] == \
            [cell.strip().lower() for cell in expected.split(",")]:
        return " — header matches the template ✅"
    return " — header differs from the template ⚠️ (intentional? keep it; otherwise re-download the template in Step 2)"


def render_checklist():
    rows = []
    for required, (purpose, spec) in REQUIRED_FILES.items():
        name = _find_uploaded(required)
        if name:
            rows.append("<tr><td>✅</td><td><code>" + required + "</code></td><td>"
                        + purpose + "</td><td><code>" + spec + "</code></td><td>"
                        + name + _header_note(required, name) + "</td></tr>")
        else:
            rows.append("<tr><td>⬜</td><td><code>" + required + "</code></td><td>"
                        + purpose + "</td><td><code>" + spec + "</code></td>"
                        "<td>— not uploaded —</td></tr>")
    th = "padding:6px 10px;border:1px solid #d0d7de;background:#f6f8fa;text-align:left;"
    td = "padding:6px 10px;border:1px solid #d0d7de;"
    checklist.value = (
        "<table style='border-collapse:collapse;font-size:13px;margin:8px 0'>"
        "<tr><th style='" + th + "'>Status</th><th style='" + th + "'>Required file</th>"
        "<th style='" + th + "'>Purpose</th><th style='" + th + "'>Expected header / content</th>"
        "<th style='" + th + "'>Uploaded as</th></tr>" + "".join(rows) + "</table>"
    )


def render_preview(_clicked=None):
    with preview_output:
        clear_output()
        if not UPLOADED_FILES:
            print("⬜ Nothing uploaded yet.")
            print("   Download the templates in Step 2, fill them in, then use 📁 Select Files.")
            print("   In a hurry? Skip to Step 4 — a bundled demo dataset is validated automatically.")
            return
        print("📦 " + str(len(UPLOADED_FILES)) + " file(s) uploaded\n" + "=" * 70)
        for filename, content in UPLOADED_FILES.items():
            size_kb = len(content.encode("utf-8")) / 1024
            lines = content.splitlines()
            print("\n📄 " + filename + " (" + format(size_kb, ".2f") + " KB, "
                  + str(len(lines)) + " lines)")
            print("-" * 70)
            for number, line in enumerate(lines[:5], 1):
                print(format(number, "3d") + " | " + line)
            if len(lines) > 5:
                print("   … (" + str(len(lines) - 5) + " more lines)")
        print("\n" + "=" * 70 + "\n📋 REQUIRED-FILES CHECKLIST\n" + "=" * 70)
        complete = True
        for required, (purpose, spec) in REQUIRED_FILES.items():
            name = _find_uploaded(required)
            found = name is not None
            note = (name + _header_note(required, name)) if found else "missing"
            print(("  ✅ " if found else "  ❌ ") + required.ljust(14) + " — " + purpose
                  + "  [" + note + "]")
            complete = complete and found
        print("\n🎉 All required files present — continue to Step 4." if complete
              else "\n⚠️ Upload the missing files, or skip to Step 4 for the auto demo.")


def on_upload_selected(change):
    new_value = change.get("new") if isinstance(change, dict) else None
    if not new_value:
        return
    if isinstance(new_value, dict):  # ipywidgets 7.x dict shape
        items = list(new_value.items())
    else:                            # ipywidgets 8.x tuple shape
        items = [(info.get("name", "file_" + str(i)), info) for i, info in enumerate(new_value)]
    for filename, info in items:
        UPLOADED_FILES[filename] = _decode(info.get("content", ""))
    render_checklist()
    render_preview()


upload_widget.observe(on_upload_selected, names="value")
preview_button.on_click(render_preview)
render_checklist()
display(HTML("<h3>📋 Files this notebook expects (live checklist)</h3>"), checklist,
        HTML("<h3>📤 Upload your files — preview is automatic</h3>"),
        upload_widget, preview_button, preview_output)


## ✅ Step 4 — Run Validation

The audit **runs automatically** when this cell executes: uploaded files take
priority, and if nothing has been uploaded the bundled demo dataset
(`sentinel2_l2a`) is validated instead so *Run all* always finishes green.
The checks mirror the repository inspector — required fields, strict semver,
approved licenses, parseable data, checksums — and re-running the cell (or
clicking the button) re-audits whatever is currently uploaded.

In [ ]:
APPROVED_LICENSES = {"MIT", "CC-BY-4.0", "CC-BY-SA-4.0", "CC0-1.0"}
REQUIRED_FIELDS = ("asset_id", "title", "version", "custodian", "license_ref", "abstract")
SEMVER = re.compile(r"^\d+\.\d+\.\d+$")
EXPECTED_HEADERS = globals().get("EXPECTED_HEADERS") or {"data.csv": "scene_id,acq_date,sensor,cloud_pct,crs,resolution_m"}

# Bundled demo dataset (the flavor's first registered asset). When nothing
# has been uploaded yet, the audit validates this so Runtime → Run all
# finishes green end-to-end even before you touch Step 3.
DEMO_FILES = {
    "record.yaml": """asset_id: sentinel2_l2a
title: "Sentinel-2 L2A Scene Register Q1 2026"
version: "1.0.0"
custodian: "Syed Rafi Ahmed"
license_ref: CC-BY-4.0
abstract: >
  Sentinel-2 L2A Scene Register Q1 2026 — curated, versioned and checksummed via GeoVault.
keywords: ["sentinel-2", "remote-sensing", "surface-reflectance"]
format: "GeoTIFF + CSV scene index"
provenance:
  source: "Original research collection, 2026"
  processing: "Registered and integrity-checked at scaffold time"
""",
    "data.csv": """scene_id,acq_date,sensor,cloud_pct,crs,resolution_m
S2_43PGQ_20260112,2026-01-12,MSI,12.4,EPSG:32643,10
S2_43PGQ_20260201,2026-02-01,MSI,8.1,EPSG:32643,10
S2_43PGQ_20260221,2026-02-21,MSI,21.7,EPSG:32643,10
""",
    "LICENSE.txt": """Dataset: Sentinel-2 L2A Scene Register Q1 2026
Asset ID: sentinel2_l2a
Version: 1.0.0
Custodian: Syed Rafi Ahmed
License: CC-BY-4.0

This dataset is distributed under the CC-BY-4.0 license listed in the
institutional rights catalog (policies/rights_catalog.yaml) of GeoVault.
Reuse requires attribution as defined by that license; consult the catalog
entry for the canonical license text and redistribution terms.
""",
}


class DatasetValidator:
    """Mirrors the repository inspector: metadata, license, data, checksums."""

    def __init__(self, files):
        self.files = files
        self.metadata = {}
        self.issues = []
        self.warnings = []
        self.checksums = {}

    def _find(self, needle):
        for name in self.files:
            if needle.lower() in name.lower():
                return name
        return None

    def validate(self):
        self._check_required_files()
        self._check_metadata()
        self._check_license()
        self._check_data()
        self._build_checksums()
        return not self.issues

    def _check_required_files(self):
        for required in REQUIRED_FILES:
            if self._find(required) is None:
                self.issues.append("Missing required file: " + required)

    def _check_metadata(self):
        name = self._find("record.yaml")
        if name is None:
            return
        try:
            self.metadata = yaml.safe_load(self.files[name]) or {}
        except yaml.YAMLError as error:
            mark = getattr(error, "problem_mark", None)
            where = " (line " + str(mark.line + 1) + ")" if mark is not None else ""
            self.issues.append("record.yaml: invalid YAML syntax" + where)
            return
        for field in REQUIRED_FIELDS:
            value = self.metadata.get(field)
            if value is None or (isinstance(value, str) and not value.strip()):
                self.issues.append("Metadata missing required field: " + field)
        version = str(self.metadata.get("version", ""))
        if version and not SEMVER.match(version):
            self.issues.append("Invalid semantic version: " + version
                               + " (expected MAJOR.MINOR.PATCH)")
        license_ref = str(self.metadata.get("license_ref", ""))
        if license_ref and license_ref not in APPROVED_LICENSES:
            # Mirrors the repository inspector, which raises an ISSUE (not a
            # warning) for licenses outside policies/rights_catalog.yaml —
            # so a dataset that passes here really does pass CI.
            self.issues.append("License " + license_ref
                               + " is not approved in the rights catalog")

    def _check_license(self):
        name = self._find("LICENSE.txt")
        if name is not None and len(self.files[name].strip()) < 40:
            self.warnings.append("LICENSE.txt looks very short — include the full license text")

    def _check_data(self):
        name = self._find("data.csv")
        if name is None:
            return
        try:
            rows = [row for row in csv.reader(io.StringIO(self.files[name])) if row]
        except csv.Error as error:
            self.issues.append("data.csv: cannot parse as CSV — " + str(error))
            return
        if len(rows) < 2:
            self.issues.append("data.csv needs a header row and at least one data row")
            return
        expected = EXPECTED_HEADERS.get("data.csv")
        if expected:
            actual = [cell.strip().lower() for cell in rows[0]]
            wanted = [cell.strip().lower() for cell in expected.split(",")]
            if actual != wanted:
                self.warnings.append(
                    "data.csv header differs from the template (expected: "
                    + expected + ") — intentional extra columns are fine")

    def _build_checksums(self):
        for name, content in sorted(self.files.items()):
            if name.lower() in SKIP_CHECKSUM_FILES:
                continue
            self.checksums[name] = hashlib.sha256(content.encode("utf-8")).hexdigest()

    def report(self):
        return {
            "passed": not self.issues,
            "issues": self.issues,
            "warnings": self.warnings,
            "metadata": self.metadata,
            "checksums": self.checksums,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        }


validate_button = widgets.Button(description="🚀 Run Validation", button_style="success")
validate_output = widgets.Output()


def _plain_text_report(report):
    """Human-readable audit summary (auto-saved + downloadable as .txt)."""
    lines = ["GeoVault compliance report",
             "generated: " + report["timestamp"],
             "-" * 60,
             "result: " + ("PASSED" if report["passed"] else "FAILED"),
             "issues: " + str(len(report["issues"]))
             + " · warnings: " + str(len(report["warnings"]))]
    for number, issue in enumerate(report["issues"], 1):
        lines.append("  " + str(number) + ". " + issue)
    for number, warning in enumerate(report["warnings"], 1):
        lines.append("  W" + str(number) + ". " + warning)
    metadata = report.get("metadata") or {}
    for key in ("asset_id", "title", "version", "custodian", "license_ref"):
        if metadata.get(key):
            lines.append(key + ": " + str(metadata[key]))
    lines.append("checksummed files: " + str(len(report["checksums"])))
    return "\n".join(lines) + "\n"


def _audit_summary_html(report):
    """Standalone pass/fail summary card (auto-saved as audit_summary.html)."""
    ok = bool(report["passed"])
    color = "#1a7f37" if ok else "#cf222e"
    badge = "✅ PASSED" if ok else "🚨 FAILED"
    metadata = report.get("metadata") or {}
    rows = "".join("<tr><td>" + key + "</td><td>" + str(metadata.get(key, "—"))
                   + "</td></tr>" for key in
                   ("asset_id", "title", "version", "custodian", "license_ref"))
    subject = str(metadata.get("asset_id") or "dataset")
    return ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
            "<title>GeoVault audit summary</title>"
            "<style>body{font-family:-apple-system,'Segoe UI',sans-serif;background:#0d1117;"
            "color:#c9d1d9;display:flex;min-height:100vh;align-items:center;justify-content:center;"
            "margin:0}.card{background:#161b22;border:1px solid #30363d;border-radius:12px;"
            "padding:28px 36px;min-width:340px}h1{margin:0 0 12px;font-size:1.3rem}"
            ".badge{color:#fff;background:" + color + ";display:inline-block;padding:6px 14px;"
            "border-radius:999px;font-weight:700}table{border-collapse:collapse;margin-top:16px;"
            "font-size:.9rem}td{padding:4px 12px 4px 0}td:first-child{color:#8b949e}"
            ".meta{color:#8b949e;font-size:.8rem;margin-top:14px}</style></head><body>"
            "<div class='card'><h1>🌍 GeoVault</h1>"
            "<span class='badge'>" + badge + "</span>"
            "<table>" + rows + "</table>"
            "<p class='meta'>audited " + subject + " · "
            + str(len(report["checksums"])) + " file(s) checksummed · "
            + report["timestamp"] + "<br>"
            "Refresh dashboard.html — its ‘Load live audit report’ button reads "
            "compliance_report.json.</p></div></body></html>\n")


def _auto_save_artifacts(report):
    """Write the four audit artifacts used by the dashboard-refresh flow."""
    Path("compliance_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    Path("compliance_report.txt").write_text(
        _plain_text_report(report), encoding="utf-8")
    Path("audit_summary.html").write_text(
        _audit_summary_html(report), encoding="utf-8")
    lines = ["# SHA-256 integrity manifest", "# Auto-generated by GeoVault"]
    for name, digest in report["checksums"].items():
        lines.append(digest + "  " + name)
    Path("integrity.sha256").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("💾 Auto-saved: compliance_report.json · compliance_report.txt · "
          "integrity.sha256 · audit_summary.html")
    if IN_COLAB and files is not None and report["passed"]:
        files.download("compliance_report.json")


def active_source():
    """Uploaded files take priority; otherwise the bundled demo dataset runs."""
    uploaded = globals().get("UPLOADED_FILES") or {}
    if uploaded:
        return dict(uploaded), False
    return dict(DEMO_FILES), True


def run_validation(_clicked=None):
    with validate_output:
        clear_output()
        source, is_demo = active_source()
        if is_demo:
            print("🧪 Demo mode: nothing uploaded yet — validating the bundled demo "
                  "dataset (sentinel2_l2a) so you can see the full workflow.")
            print("   Upload your own files in Step 3, then re-run this cell or "
                  "click the button.\n")
        print("🔍 Running GeoVault compliance audit…\n")
        validator = DatasetValidator(source)
        validator.validate()
        report = validator.report()
        globals()["LAST_REPORT"] = report
        _auto_save_artifacts(report)
        if report["passed"]:
            display(HTML("<div style='background:#1a7f37;color:#fff;padding:16px;border-radius:8px;'>"
                         "<h2 style='margin:0;'>✅ AUDIT PASSED</h2>"
                         "<p style='margin:4px 0 0;'>Your dataset satisfies the stewardship policy.</p></div>"))
        else:
            display(HTML("<div style='background:#cf222e;color:#fff;padding:16px;border-radius:8px;'>"
                         "<h2 style='margin:0;'>🚨 AUDIT FAILED</h2>"
                         "<p style='margin:4px 0 0;'>Fix the issues below, then re-run.</p></div>"))
        metadata = report["metadata"]
        if metadata:
            print("\n📋 METADATA SUMMARY\n" + "-" * 70)
            for key in ("asset_id", "title", "version", "custodian", "license_ref"):
                print("  " + key.ljust(14) + ": " + str(metadata.get(key, "—")))
        if report["issues"]:
            print("\n❌ ISSUES (" + str(len(report["issues"])) + ")\n" + "-" * 70)
            for number, issue in enumerate(report["issues"], 1):
                print("  " + str(number) + ". " + issue)
        if report["warnings"]:
            print("\n⚠️ WARNINGS (" + str(len(report["warnings"])) + ")\n" + "-" * 70)
            for number, warning in enumerate(report["warnings"], 1):
                print("  " + str(number) + ". " + warning)
        if report["checksums"]:
            print("\n🔐 SHA-256 CHECKSUMS\n" + "-" * 70)
            for name, digest in report["checksums"].items():
                print("  " + digest[:16] + "…  " + name)
        print("\n📅 Completed: " + report["timestamp"])
        if is_demo and report["passed"]:
            print("\n💡 That was the bundled demo dataset — upload your own files "
                  "in Step 3 and re-run this cell to audit your data.")


validate_button.on_click(run_validation)
display(HTML("<h3>🚀 Validate your dataset</h3>"), validate_button, validate_output)

# ▶️ Auto-run: the audit executes immediately when the cell runs (uploaded
# files first, bundled demo otherwise), so Runtime → Run all completes
# end-to-end without clicking anything.
run_validation(None)


## 📊 Step 5 — Results & Dashboard Refresh

Every audit **auto-saves** four artifacts into the working directory (on
Colab: click the 📁 folder icon in the left sidebar to fetch them):

- `compliance_report.json` — machine-readable audit (what `dashboard.html`
  reads via *Load live audit report*)
- `compliance_report.txt` — human-readable audit summary
- `integrity.sha256` — ready-to-commit checksum manifest
- `audit_summary.html` — standalone pass/fail summary card

The buttons below re-download them on demand; after a **passing** Colab
audit the JSON report downloads automatically.

> ♻️ **Dashboard refresh:** commit `compliance_report.json` next to
> `dashboard.html` and its live-audit panel renders the latest result —
> no other change needed.

In [ ]:
report_button = widgets.Button(description="📥 Download compliance_report.json", button_style="info")
txt_button = widgets.Button(description="📥 Download compliance_report.txt", button_style="info")
summary_button = widgets.Button(description="📥 Download audit_summary.html", button_style="info")
manifest_button = widgets.Button(description="📥 Download integrity.sha256", button_style="info")
download_output = widgets.Output()


def download_report(_clicked):
    with download_output:
        clear_output()
        report = globals().get("LAST_REPORT")
        if not report:
            print("⚠️ Run validation first (Step 4).")
            return
        save_bytes("compliance_report.json", json.dumps(report, indent=2, ensure_ascii=False))
        print("✅ Downloaded: compliance_report.json")


def download_manifest(_clicked):
    with download_output:
        clear_output()
        report = globals().get("LAST_REPORT")
        if not report:
            print("⚠️ Run validation first (Step 4).")
            return
        lines = ["# SHA-256 integrity manifest", "# Auto-generated by GeoVault"]
        for name, digest in report["checksums"].items():
            lines.append(digest + "  " + name)
        save_bytes("integrity.sha256", "\n".join(lines) + "\n")
        print("✅ Downloaded: integrity.sha256")


def download_txt(_clicked):
    with download_output:
        clear_output()
        report = globals().get("LAST_REPORT")
        if not report:
            print("⚠️ Run validation first (Step 4).")
            return
        save_bytes("compliance_report.txt", _plain_text_report(report))
        print("✅ Downloaded: compliance_report.txt")


def download_summary(_clicked):
    with download_output:
        clear_output()
        report = globals().get("LAST_REPORT")
        if not report:
            print("⚠️ Run validation first (Step 4).")
            return
        save_bytes("audit_summary.html", _audit_summary_html(report))
        print("✅ Downloaded: audit_summary.html")


for button, handler in ((report_button, download_report), (txt_button, download_txt),
                        (summary_button, download_summary), (manifest_button, download_manifest)):
    button.on_click(handler)
display(HTML("<h3>📊 Download results</h3>"), report_button, txt_button,
        summary_button, manifest_button, download_output)


---

## 🎉 Done!

Your dataset has been validated against FAIR-aligned stewardship policy.

### What to do next

1. ✅ Fix any issues reported in Step 4 and re-run the audit
2. 📥 Download `integrity.sha256` and `compliance_report.json`
3. 📁 Place `data.csv`, `record.yaml`, `LICENSE.txt` and `integrity.sha256`
   together in one folder
4. ♻️ Drop `compliance_report.json` next to `dashboard.html` to refresh
   its live-audit panel
5. 🚀 Push the folder to GitHub — the Compliance Gate workflow validates it
   automatically

> 🧪 If the Step 4 banner said *Demo mode*, the results above describe the
> bundled demo dataset — re-run with your own uploads before using the
> downloads.

---

*Built with GeoVault v1.3.0 — Remote Sensing, GIS & ML Data Stewardship Pipeline*
